In [1]:
import os
from itertools import combinations
from gurobipy import GRB
import sys
sys.path.insert(0, '../../scripts')
import post_processing as ppr

input_dir = '../../../plasbin_flow_input'
output_dir = '../../../organized_data/output'
gc_file = '../../../organized_data/gc_intervals.txt'
results_dir = 'post_processing_results'

The purpose of this notebook is to run an intial test of our post-processing MILP to ensure it is functioning as intended and to get a quick idea of what the output looks like.

Testing on one sample:

In [2]:
model = ppr.post_processing_model('sample_15', 'pbf_P14', 'pbf_P15', '../../../plasbin_flow_input', '../../../organized_data/output', '../../../organized_data/gc_intervals.txt')
model.optimize()

Set parameter Username
Academic license - for non-commercial use only - expires 2024-06-27
Gurobi Optimizer version 10.0.2 build v10.0.2rc0 (win64)

CPU model: AMD Ryzen 5 3500U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5604 rows, 2395 columns and 13824 nonzeros
Model fingerprint: 0xaa4fbb87
Variable types: 404 continuous, 1991 integer (1991 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+04]
  Objective range  [5e-05, 2e+00]
  Bounds range     [1e+00, 1e+04]
  RHS range        [1e+00, 1e+04]
Presolve removed 4562 rows and 1386 columns
Presolve time: 0.10s
Presolved: 1042 rows, 1009 columns, 3571 nonzeros
Variable types: 229 continuous, 780 integer (780 binary)
Found heuristic solution: objective 3.2250610

Root relaxation: objective 6.335915e+00, 523 iterations, 0.02 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      

Looking at non-zero variables in the solution, while sorting the `q` variables to highlight the edge order of the path:

In [3]:
nzX = 0
nzQ = 0
nzZ = 0
nzGC = 0
flow_dict = {}
for v in model.getVars():
    if v.X > 0.0001:
        if v.VarName[0] == 'x':
            nzX += 1
            print('%s %g' % (v.VarName, v.X))
        elif v.VarName[0] == 'q':
            nzQ += 1
            flow_dict[v.VarName] = v.X
        elif v.VarName[0] == 'z':
            nzZ += 1
            print('%s %g' % (v.VarName, v.X))
        elif v.VarName[0] == 'c':
            nzGC += 1
            print('%s %g' % (v.VarName, v.X))
            
for key in sorted(flow_dict, key=flow_dict.get, reverse=True):
    print(key, flow_dict[key])
print('\nObj: %g' % model.ObjVal)
for d in [var for var in model.getVars() if var.VarName == 'd']:
    print('minimum read depth:', d.X, '\n')
print('non-zero x:', nzX)
print('non-zero q:', nzQ)
print('non-zero z:', nzZ)
print('non-zero ctg_GC:', nzGC)

x[29,246] 1
x[s,121] 1
x[114,t] 1
x[121,125] 1
x[125,133] 1
x[130,114] 1
x[133,225] 1
x[137,236] 1
x[165,231] 1
x[180,130] 1
x[187,29] 1
x[215,187] 1
x[225,137] 1
x[231,215] 1
x[236,165] 1
x[246,180] 1
z[29] 1
z[s] 1
z[114] 1
z[121] 1
z[125] 1
z[130] 1
z[133] 1
z[t] 1
z[137] 1
z[165] 1
z[180] 1
z[187] 1
z[215] 1
z[225] 1
z[231] 1
z[236] 1
z[246] 1
ctg_GC[29,1] 1
ctg_GC[s,1] 1
ctg_GC[114,1] 1
ctg_GC[121,1] 1
ctg_GC[125,1] 1
ctg_GC[130,1] 1
ctg_GC[133,1] 1
ctg_GC[t,1] 1
ctg_GC[137,1] 1
ctg_GC[165,1] 1
ctg_GC[180,1] 1
ctg_GC[187,1] 1
ctg_GC[215,1] 1
ctg_GC[225,1] 1
ctg_GC[231,1] 1
ctg_GC[236,1] 1
ctg_GC[246,1] 1
q[s,121] 16.0000000334711
q[121,125] 15.000000033444046
q[125,133] 14.00000003341699
q[133,225] 13.000000033389934
q[225,137] 12.000000033247321
q[137,236] 11.000000033104708
q[236,165] 10.000000032962095
q[165,231] 9.000000033104708
q[231,215] 8.000000033104708
q[215,187] 7.000000033104708
q[187,29] 6.000000000696117
q[29,246] 5.0000000005832534
q[246,180] 4.000000000464059
q[180

We will now try running our post-processing on pairs of predicted bins which are:
- Properly contained in a ground truth plasmid
- Disjoint

The solution files are in USRA_PLASMIDS_2023/data/POST-PROCESS-TEST-2023-07-16/post_processing_results.

In [7]:
sample_dict = {}
model_dict = {}

with open('../../test/split_bins.tsv', 'r') as bins:
    line = next(bins, None)
    while line:
        lst = line[:-1].split(sep='\t')
        if lst[3] != '' and lst[0] != 'SAMPLE':
            pls_lst = lst[3][1:-1].split(sep=',')
            ctg_lsts = [pls[1:-1].split(sep=',') for pls in lst[4][1:-1].split(sep=';')]
            for ctg_lst in ctg_lsts:
                ctg_lsts[ctg_lsts.index(ctg_lst)] = [ctg.split(sep=':')[0] for ctg in ctg_lst]
            sample_dict[lst[1]] = (lst[0], pls_lst, ctg_lsts)
        line = next(bins, None)

for key in sample_dict:
    if len(sample_dict[key][1]) > 1:
        for subset in combinations(sample_dict[key][1], 2):
            ind0 = sample_dict[key][1].index(subset[0])
            ind1 = sample_dict[key][1].index(subset[1])
            if set(sample_dict[key][2][ind0]) & set(sample_dict[key][2][ind1]) == set():
                model = ppr.post_processing_model(sample_dict[key][0], *subset, input_dir, output_dir, gc_file)
                model.setParam('OutputFlag', False)
                model.optimize()
                filename = '_'.join([sample_dict[key][0], *subset])
                if model.getAttr('Status') != GRB.OPTIMAL:
                    with open(os.path.join(results_dir, filename + '.out'), 'w') as file:
                        file.write('Unable to solve {} {}'.format(sample_dict[key][0], subset))
                else:
                    model.write(os.path.join(results_dir, filename + '.sol'))
                print('Finished post-processing', sample_dict[key][0], subset)

Finished post-processing sample_108 ('pbf_P0', 'pbf_P2')
Finished post-processing sample_109 ('pbf_P1', 'pbf_P2')
Finished post-processing sample_112 ('pbf_P1', 'pbf_P2')
Finished post-processing sample_112 ('pbf_P1', 'pbf_P3')
Finished post-processing sample_112 ('pbf_P2', 'pbf_P3')
Finished post-processing sample_113 ('pbf_P1', 'pbf_P2')
Finished post-processing sample_117 ('pbf_P3', 'pbf_P4')
Finished post-processing sample_117 ('pbf_P3', 'pbf_P6')
Finished post-processing sample_117 ('pbf_P4', 'pbf_P6')
Finished post-processing sample_119 ('pbf_P0', 'pbf_P1')
Finished post-processing sample_120 ('pbf_P0', 'pbf_P1')
Finished post-processing sample_120 ('pbf_P0', 'pbf_P2')
Finished post-processing sample_120 ('pbf_P1', 'pbf_P2')
Finished post-processing sample_15 ('pbf_P14', 'pbf_P15')
Finished post-processing sample_15 ('pbf_P14', 'pbf_P17')
Finished post-processing sample_15 ('pbf_P14', 'pbf_P18')
Finished post-processing sample_15 ('pbf_P14', 'pbf_P22')
Finished post-processing sa

Finished post-processing sample_44 ('pbf_P3', 'pbf_P4')
Finished post-processing sample_44 ('pbf_P3', 'pbf_P5')
Finished post-processing sample_44 ('pbf_P3', 'pbf_P6')
Finished post-processing sample_44 ('pbf_P4', 'pbf_P5')
Finished post-processing sample_44 ('pbf_P4', 'pbf_P6')
Finished post-processing sample_44 ('pbf_P5', 'pbf_P6')
Finished post-processing sample_45 ('pbf_P0', 'pbf_P1')
Finished post-processing sample_45 ('pbf_P0', 'pbf_P2')
Finished post-processing sample_45 ('pbf_P0', 'pbf_P3')
Finished post-processing sample_45 ('pbf_P0', 'pbf_P4')
Finished post-processing sample_45 ('pbf_P0', 'pbf_P5')
Finished post-processing sample_45 ('pbf_P0', 'pbf_P6')
Finished post-processing sample_45 ('pbf_P0', 'pbf_P7')
Finished post-processing sample_45 ('pbf_P1', 'pbf_P2')
Finished post-processing sample_45 ('pbf_P1', 'pbf_P3')
Finished post-processing sample_45 ('pbf_P1', 'pbf_P4')
Finished post-processing sample_45 ('pbf_P1', 'pbf_P5')
Finished post-processing sample_45 ('pbf_P1', 'p

In [5]:
for file in os.listdir(results_dir):
    f = os.path.join(results_dir, file)
    if file[-4:] == '.sol':
        print('Post-processing results for', file[:-4], ':')
        ppr.print_post_processing_sol_file(f)
        print('\n')
    elif file[-4:] == '.out':
        with open(f, 'r') as out:
            print(out.read())
            print('\n')

Post-processing results for sample_108_pbf_P0_pbf_P2 :
Optimal path (as defined by flow values):
q[s,t] 1.0

Minimum read depth: 1.5300000000006548
Objective value: 3.6557557514802266


Post-processing results for sample_109_pbf_P1_pbf_P2 :
Optimal path (as defined by flow values):
q[s,t] 1.0

Minimum read depth: 4.860000000000582
Objective value: 8.208800385431646


Post-processing results for sample_112_pbf_P1_pbf_P2 :
Optimal path (as defined by flow values):
q[s,89] 3.0
q[89,109] 2.0
q[109,t] 1.0

Minimum read depth: 1.6244995194538205
Objective value: 2.2469949814917882


Post-processing results for sample_112_pbf_P1_pbf_P3 :
Optimal path (as defined by flow values):
q[s,t] 1.0

Minimum read depth: 1.7000000000007276
Objective value: 3.7718097499324754


Post-processing results for sample_112_pbf_P2_pbf_P3 :
Optimal path (as defined by flow values):
q[s,t] 1.0

Minimum read depth: 1.7000000000007276
Objective value: 3.738768949019048


Post-processing results for sample_113_pbf_P1

Optimal path (as defined by flow values):
q[s,t] 1.0

Minimum read depth: 11.340000000000146
Objective value: 12.183134764458874


Post-processing results for sample_24_pbf_P0_pbf_P6 :
Optimal path (as defined by flow values):
q[s,220] 6.999999999997669
q[220,217] 6.000000000006029
q[217,163] 5.000000000002276
q[163,262] 4.000000000001707
q[262,146] 3.000000000000343
q[146,216] 2.0
q[216,t] 1.0

Minimum read depth: 6.197981461746167
Objective value: 4.529720502473267


Post-processing results for sample_24_pbf_P0_pbf_P8 :
Optimal path (as defined by flow values):
q[s,220] 3.0
q[220,217] 2.0
q[217,t] 1.0

Minimum read depth: 6.197981461746167
Objective value: 5.863883334651012


Post-processing results for sample_24_pbf_P0_pbf_P9 :
Optimal path (as defined by flow values):
q[s,149] 4.0
q[149,179] 3.0
q[179,202] 2.0
q[202,t] 1.0

Minimum read depth: 8.3799999999992
Objective value: 7.723346179572275


Post-processing results for sample_24_pbf_P10_pbf_P13 :
Optimal path (as defined by flo

Optimal path (as defined by flow values):
q[s,132] 5.0
q[132,207] 4.0
q[207,168] 3.0
q[168,208] 2.0
q[208,t] 1.0

Minimum read depth: 0.31551913783732743
Objective value: 2.163450381834806


Post-processing results for sample_25_pbf_P6_pbf_P7 :
Optimal path (as defined by flow values):
q[s,132] 4.999999999995964
q[132,207] 3.999999999995552
q[207,157] 2.999999999993719
q[157,189] 2.0
q[189,t] 1.0

Minimum read depth: 0.31551913783732743
Objective value: 1.2083426952096838


Post-processing results for sample_25_pbf_P7_pbf_P10 :
Optimal path (as defined by flow values):
q[s,189] 6.000000000004893
q[189,157] 4.99999999999516
q[157,207] 3.999999999995992
q[207,168] 2.999999999996994
q[168,208] 1.999999999997996
q[208,t] 0.999999999998998

Minimum read depth: 3.0799999999999272
Objective value: 1.9247044330550684


Post-processing results for sample_25_pbf_P7_pbf_P11 :
Optimal path (as defined by flow values):
q[s,189] 5.0
q[189,157] 4.0
q[157,207] 3.0
q[207,132] 2.0
q[132,t] 1.0

Minimum 

Optimal path (as defined by flow values):
q[s,114] 23.0
q[114,216] 22.0
q[216,59] 21.0
q[59,253] 20.0
q[253,57] 19.0
q[57,120] 18.0
q[120,131] 17.0
q[131,173] 16.0
q[173,102] 15.0
q[102,56] 14.0
q[56,179] 13.0
q[179,154] 12.0
q[154,81] 11.0
q[81,285] 10.0
q[285,136] 9.0
q[136,268] 8.0
q[268,187] 7.0
q[187,197] 6.0
q[197,190] 5.0
q[190,98] 4.0
q[98,89] 3.0
q[89,137] 2.0
q[137,t] 1.0

Minimum read depth: 0.058304240965299285
Objective value: 4.426203102788729


Post-processing results for sample_33_pbf_P14_pbf_P17 :
Optimal path (as defined by flow values):
q[s,309] 27.0
q[309,175] 26.0
q[175,180] 25.0
q[180,117] 24.0
q[117,86] 23.0
q[86,99] 22.0
q[99,45] 21.0
q[45,253] 20.0
q[253,57] 19.0
q[57,120] 18.0
q[120,131] 17.0
q[131,173] 16.0
q[173,102] 15.0
q[102,56] 14.0
q[56,179] 13.0
q[179,154] 12.0
q[154,81] 11.0
q[81,285] 10.0
q[285,136] 9.0
q[136,268] 8.0
q[268,187] 7.0
q[187,197] 6.0
q[197,190] 5.0
q[190,98] 4.0
q[98,89] 3.0
q[89,137] 2.0
q[137,t] 1.0

Minimum read depth: 0.058304240965

Optimal path (as defined by flow values):
q[s,332] 14.999999999999993
q[332,338] 13.999999999999993
q[338,306] 12.999999999999993
q[306,457] 11.999999999999993
q[457,212] 11.000000000000044
q[212,249] 10.000000000000044
q[249,202] 9.000000000000044
q[202,414] 8.000000000000046
q[414,396] 7.0
q[396,304] 5.999999999999969
q[304,76] 4.999999999999969
q[76,477] 3.999999999999969
q[477,131] 3.0
q[131,456] 2.0
q[456,t] 1.0

Minimum read depth: 0.31999999999970896
Objective value: 1.0806021289598355


Unable to solve sample_44 ('pbf_P1', 'pbf_P5')


Post-processing results for sample_44_pbf_P1_pbf_P6 :
Optimal path (as defined by flow values):
q[s,332] 4.0
q[332,464] 3.0
q[464,444] 2.0
q[444,t] 1.0

Minimum read depth: 0.31999999999970896
Objective value: 1.0215201685300388


Unable to solve sample_44 ('pbf_P3', 'pbf_P4')


Unable to solve sample_44 ('pbf_P3', 'pbf_P5')


Unable to solve sample_44 ('pbf_P3', 'pbf_P6')


Unable to solve sample_44 ('pbf_P4', 'pbf_P5')


Post-processing results 

Optimal path (as defined by flow values):
q[s,361] 3.0
q[361,227] 2.0
q[227,t] 1.0

Minimum read depth: 0.3999999999996362
Objective value: 0.948518405889732


Post-processing results for sample_48_pbf_P0_pbf_P5 :
Optimal path (as defined by flow values):
q[s,104] 9.0
q[104,401] 8.0
q[401,57] 7.0
q[57,205] 6.0
q[205,171] 5.0
q[171,404] 4.0
q[404,45] 3.0
q[45,278] 2.0
q[278,t] 1.0

Minimum read depth: 0.3999999999996362
Objective value: -0.47748357188041557


Post-processing results for sample_48_pbf_P1_pbf_P2 :
Optimal path (as defined by flow values):
q[s,57] 3.0
q[57,401] 2.0
q[401,t] 1.0

Minimum read depth: 0.42000000000007276
Objective value: 0.4443154627497454


Post-processing results for sample_48_pbf_P1_pbf_P3 :
Optimal path (as defined by flow values):
q[s,171] 3.0
q[171,404] 2.0
q[404,t] 1.0

Minimum read depth: 0.42000000000007276
Objective value: 0.48544098116281065


Post-processing results for sample_48_pbf_P1_pbf_P4 :
Optimal path (as defined by flow values):
q[s,171] 4